# Supply Chain Assistant with Oracle AI Agent Memory (OCI)

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

**Framework:** OpenAI Agents SDK over OCI Generative AI · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook builds a **supply chain assistant** that tracks and updates shipment cargo. The assistant exposes tools to read and mutate shipment state and uses Oracle AI Agent Memory to remember facts about shipments, customers, and operational context across sessions.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational, tool-using | Operations support, customer service, internal ops copilots |
| **Deep Research** | Multi-step autonomous investigation | Literature review, scoping, market research |
| **Workflow** | Predetermined sequence with conditional branches | Approval pipelines, compliance gates, structured triage |

> **This notebook focuses on the Assistant mode.**
>
> An assistant agent is conversational first: it responds to human prompts, calls tools when it needs to read or change state, and carries context across turns. Memory is how it stays coherent across a working day or across sessions with the same operator.


## What You'll Learn

- How to define **in-process tools** with the Claude Agent SDK using the `@tool` decorator
- How to wire those tools into an **MCP server** that the Claude Agent SDK can serve
- How to run the assistant in **multi-turn** mode with `ClaudeSDKClient`
- How to use Oracle AI Agent Memory to persist shipment records, operational notes, and conversation history so the assistant stays coherent across restarts

> **💡 Key insight:** In the Claude Agent SDK, the `@tool` decorator + `create_sdk_mcp_server` pattern is how Python functions become things the LLM can call. You don't define tools inline on the agent; you register them as an MCP server and point the agent at it.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **OCI Generative AI** access to `xai.grok-3-fast`
- Local OCI config file at `~/.oci/config`, or set `OCI_CONFIG_FILE` and `OCI_PROFILE`
- **`OCI_GENAI_COMPARTMENT_OCID`** for native OCI mode, or a `compartment_id`/`tenancy` value in the selected OCI profile
- The `oracleagentmemory` wheel installed in the active kernel's environment


## 1. Install dependencies

In [ ]:
%pip install -q oracleagentmemory==26.4.0 openai-agents openai oci nest_asyncio


## 2. Configure OCI and Oracle connection


In [ ]:
import configparser
import os
from pathlib import Path


def _expand_path(value: str) -> str:
    return os.path.expanduser(os.path.expandvars(value))


def _read_oci_profile_region(config_file: str, profile_name: str) -> str:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return ""

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT" and parser.defaults().get("region"):
        return parser.defaults()["region"].strip()
    for section in (profile_name, f"profile {profile_name}"):
        if parser.has_section(section) and parser.has_option(section, "region"):
            return parser.get(section, "region").strip()
    return ""


def _read_oci_profile_values(config_file: str, profile_name: str) -> dict[str, str]:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return {}

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT":
        values = dict(parser.defaults())
    else:
        values = {}
        for section in (profile_name, f"profile {profile_name}"):
            if parser.has_section(section):
                values.update(dict(parser.items(section)))
                break

    if values.get("key_file"):
        values["key_file"] = _expand_path(values["key_file"])
    return values


def _set_oci_env_aliases(config_file: str, profile_name: str) -> None:
    values = _read_oci_profile_values(config_file, profile_name)
    aliases = {
        "user": ("OCI_USER", "OCI_USER_ID"),
        "tenancy": ("OCI_TENANCY", "OCI_TENANCY_ID"),
        "fingerprint": ("OCI_FINGERPRINT",),
        "key_file": ("OCI_KEY_FILE",),
        "region": ("OCI_REGION", "OCI_REGION_NAME"),
    }
    for config_key, env_names in aliases.items():
        value = values.get(config_key, "")
        if not value:
            continue
        for env_name in env_names:
            os.environ.setdefault(env_name, value)


def _env(name: str, default: str = "") -> str:
    raw_value = os.environ.get(name)
    value = raw_value if raw_value not in (None, "") else default
    value = _expand_path(value) if name.endswith("_FILE") else value
    os.environ[name] = value
    return value


def _require_non_empty(value: str, name: str, hint: str) -> None:
    if not value:
        raise RuntimeError(f"Missing {name}. {hint}")


def _require_existing_file(value: str, name: str, hint: str) -> None:
    _require_non_empty(value, name, hint)
    if not Path(value).expanduser().exists():
        raise RuntimeError(f"{name} does not exist at {value}. {hint}")


LLM_PROVIDER = _env("LLM_PROVIDER", "oci_genai_native").strip().lower().replace("-", "_")
if LLM_PROVIDER not in {"oci_genai_native", "oci_native"}:
    raise RuntimeError("This OCI notebook supports LLM_PROVIDER=oci_genai_native only.")

LLM_MODEL = _env("LLM_MODEL", "xai.grok-3-fast").strip()
OCI_CONFIG_FILE = _env("OCI_CONFIG_FILE", "~/.oci/config")
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
OCI_PROFILE_REGION = _read_oci_profile_region(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_REGION = os.environ.get("OCI_GENAI_REGION", "").strip() or "us-chicago-1"
os.environ["OCI_GENAI_REGION"] = OCI_GENAI_REGION
OCI_GENAI_ENDPOINT = _env(
    "OCI_GENAI_ENDPOINT",
    f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
OCI_PROFILE_VALUES = _read_oci_profile_values(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_COMPARTMENT_OCID = _env(
    "OCI_GENAI_COMPARTMENT_OCID",
    OCI_PROFILE_VALUES.get("compartment_id") or OCI_PROFILE_VALUES.get("tenancy", ""),
).strip()
OCI_GENAI_MODEL_OCID = _env("OCI_GENAI_MODEL_OCID", LLM_MODEL).strip()
OCI_GENAI_EMBED_MODEL = _env("OCI_GENAI_EMBED_MODEL", "cohere.embed-english-v3.0")

_set_oci_env_aliases(OCI_CONFIG_FILE, OCI_PROFILE)
os.environ.setdefault("OCI_REGION", OCI_GENAI_REGION)
os.environ.setdefault("OCI_REGION_NAME", OCI_GENAI_REGION)
os.environ.setdefault("OCI_COMPARTMENT_ID", OCI_GENAI_COMPARTMENT_OCID)

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

_require_existing_file(
    OCI_CONFIG_FILE,
    "OCI_CONFIG_FILE",
    "Set OCI_CONFIG_FILE to your OCI config path, usually ~/.oci/config.",
)
_require_non_empty(OCI_PROFILE, "OCI_PROFILE", "Set OCI_PROFILE to a profile in OCI_CONFIG_FILE.")
_require_non_empty(OCI_GENAI_REGION, "OCI_GENAI_REGION", "Set OCI_GENAI_REGION for OCI Generative AI.")
_require_non_empty(
    OCI_GENAI_COMPARTMENT_OCID,
    "OCI_GENAI_COMPARTMENT_OCID",
    "Set this to a compartment OCID with OCI Generative AI permissions. The notebook falls back to compartment_id or tenancy from OCI_CONFIG_FILE when present.",
)
_require_non_empty(
    OCI_GENAI_MODEL_OCID,
    "OCI_GENAI_MODEL_OCID",
    "Set this to the on-demand model OCID or model ID. Defaults to LLM_MODEL, for example xai.grok-3-fast.",
)

import nest_asyncio
nest_asyncio.apply()

print("Environment configured for OCI Generative AI.")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"LLM model: {LLM_MODEL}")
print(f"OCI profile region: {OCI_PROFILE_REGION or 'not set'}")
print(f"OCI GenAI region: {OCI_GENAI_REGION}")
print(f"OCI endpoint: {OCI_GENAI_ENDPOINT}")
print(f"OCI compartment: {OCI_GENAI_COMPARTMENT_OCID[:18]}..." if OCI_GENAI_COMPARTMENT_OCID else "OCI compartment: not set")
print(f"OCI model id: {OCI_GENAI_MODEL_OCID}")
print(f"OCI embedding model: {OCI_GENAI_EMBED_MODEL}")


## 3. Connect to Oracle AI Database and initialise the memory client

The assistant uses memory for two things:

1. **Operational facts**: shipment ownership, customer preferences, known issues, stored as durable memories the assistant can recall by semantic search.
2. **Conversation history**: stored as messages on a thread so the assistant picks up where the operator left off.

This OCI variant uses native OCI SDK embeddings and a native OCI SDK LLM adapter for automatic memory extraction. No LiteLLM or hosted OpenAI key is required.

> **Key insight:** `table_name_prefix` gives each application its own namespace inside a shared Oracle schema. The supply-chain assistant's data will not collide with other OAMP-using apps in the same database.


In [ ]:
import asyncio
import json
from collections.abc import Sequence
from typing import Any

import numpy as np
import oci
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from oracleagentmemory.apis.llms.llm import ILlm, LlmResponse


class OciSdkEmbedder(IEmbedder):
    """OracleAgentMemory embedder backed directly by OCI Generative AI."""

    def __init__(self, client, compartment_id: str, model_id: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        from oci.generative_ai_inference import models

        details = models.EmbedTextDetails()
        details.inputs = texts
        details.compartment_id = self.compartment_id
        details.serving_mode = models.OnDemandServingMode(model_id=self.model_id)
        details.truncate = models.EmbedTextDetails.TRUNCATE_END
        details.input_type = (
            models.EmbedTextDetails.INPUT_TYPE_SEARCH_QUERY
            if is_query
            else models.EmbedTextDetails.INPUT_TYPE_SEARCH_DOCUMENT
        )

        response = self.client.embed_text(details)
        embeddings = getattr(response.data, "embeddings", None)
        if not embeddings:
            raise RuntimeError("OCI embed_text returned no embeddings.")
        return np.asarray(embeddings, dtype=np.float32)

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


def create_oci_genai_client():
    try:
        oci_config = oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load OCI config profile {OCI_PROFILE!r} from {OCI_CONFIG_FILE}: {exc}"
        ) from exc

    return oci.generative_ai_inference.GenerativeAiInferenceClient(
        config=oci_config,
        service_endpoint=OCI_GENAI_ENDPOINT,
        retry_strategy=oci.retry.NoneRetryStrategy(),
        timeout=(10, 240),
    )


def _message_content_to_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(getattr(item, "text", "") or getattr(item, "content", "") or item))
        return "\n".join(part for part in parts if part)
    return str(content)


def _oci_text_content(models, text: Any):
    item = models.TextContent()
    if hasattr(models.TextContent, "TYPE_TEXT"):
        item.type = models.TextContent.TYPE_TEXT
    item.text = _message_content_to_text(text)
    return item


def _chat_message_to_oci(models, message: dict):
    role = (message.get("role") or "user").lower()
    content_items = [_oci_text_content(models, message.get("content"))]
    if role == "system":
        oci_message = models.SystemMessage()
        oci_message.role = models.SystemMessage.ROLE_SYSTEM
    elif role == "assistant":
        oci_message = models.AssistantMessage()
        oci_message.role = models.AssistantMessage.ROLE_ASSISTANT
    else:
        oci_message = models.UserMessage()
        oci_message.role = models.UserMessage.ROLE_USER
    oci_message.content = content_items
    return oci_message


def _extract_oci_text(message: Any) -> str:
    parts = []
    for item in getattr(message, "content", None) or []:
        text = getattr(item, "text", None)
        if text:
            parts.append(text)
    return "\n".join(parts)


def oci_chat_text(
    messages: list[dict[str, str]],
    *,
    temperature: float = 0.2,
    max_tokens: int = 1200,
    response_json_schema: dict[str, Any] | None = None,
) -> str:
    from oci.generative_ai_inference import models

    chat_request = models.GenericChatRequest()
    chat_request.api_format = models.BaseChatRequest.API_FORMAT_GENERIC
    chat_request.messages = [_chat_message_to_oci(models, message) for message in messages]
    chat_request.max_tokens = max_tokens
    chat_request.temperature = temperature
    chat_request.top_p = 1.0
    chat_request.top_k = 0
    chat_request.is_stream = False

    if response_json_schema:
        schema = models.ResponseJsonSchema()
        schema.name = "response"
        schema.schema = response_json_schema
        schema.is_strict = False
        response_format = models.JsonSchemaResponseFormat()
        response_format.type = models.JsonSchemaResponseFormat.TYPE_JSON_SCHEMA
        response_format.json_schema = schema
        chat_request.response_format = response_format

    serving_mode = models.OnDemandServingMode()
    serving_mode.model_id = OCI_GENAI_MODEL_OCID

    chat_detail = models.ChatDetails()
    chat_detail.serving_mode = serving_mode
    chat_detail.chat_request = chat_request
    chat_detail.compartment_id = OCI_GENAI_COMPARTMENT_OCID

    response = genai_client.chat(chat_detail)
    data = response.data
    chat_response = getattr(data, "chat_response", data)
    choices = getattr(chat_response, "choices", None) or []
    message = getattr(choices[0], "message", None) if choices else None
    return _extract_oci_text(message)


class OciSdkLlm(ILlm):
    """OracleAgentMemory LLM backed directly by OCI Generative AI."""

    def __init__(self, temperature: float = 0.2, max_tokens: int = 1200):
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        if isinstance(prompt, str):
            messages = [{"role": "user", "content": prompt}]
        else:
            messages = list(prompt)
        text = oci_chat_text(
            messages,
            temperature=kwargs.get("temperature", self.temperature),
            max_tokens=kwargs.get("max_tokens", self.max_tokens),
            response_json_schema=response_json_schema,
        )
        return LlmResponse(text=text)

    async def generate_async(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        return await asyncio.to_thread(
            self.generate,
            prompt,
            response_json_schema=response_json_schema,
            **kwargs,
        )


genai_client = create_oci_genai_client()
oamp_embedder = OciSdkEmbedder(
    genai_client,
    compartment_id=OCI_GENAI_COMPARTMENT_OCID,
    model_id=OCI_GENAI_EMBED_MODEL,
)
oamp_llm = OciSdkLlm(temperature=0.2)
print("OCI Generative AI helpers ready.")

import oracledb
from oracleagentmemory.core import OracleAgentMemory

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder,
    llm=oamp_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
    table_name_prefix="supply_",
)
print("Memory client ready.")


## 4. Register the operator and the agent

Each human operator is a `user_id`, and the supply-chain assistant itself is an `agent_id`. This lets multiple operators share the same assistant with isolated memory, and lets one operator converse with multiple specialised assistants.

In [ ]:
OPERATOR_ID = "operator-morgan"
AGENT_ID = "supply-chain-assistant"

for create_fn, eid, info in [
    (memory_client.add_user, OPERATOR_ID, "Operations manager — West Coast logistics desk."),
    (memory_client.add_agent, AGENT_ID, "Supply chain assistant with read/write tools for shipment state."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

## 5. Simulate a shipment database

In production, shipment state lives in an ERP or logistics system. For this example we use an in-memory dict. The tools we expose to the agent don't care — they just need read/write methods over *some* shipment store.

> **📌 Production pattern.** Replace `SHIPMENT_DB` with calls to your real system of record (Oracle tables, an ERP API, a microservice). The tool signatures stay the same; only the bodies change.

In [ ]:
SHIPMENT_DB = {
    "SHP-1001": {
        "id": "SHP-1001", "origin": "Shanghai", "destination": "Long Beach",
        "status": "in_transit", "eta": "2026-05-02",
        "carrier": "Maersk", "containers": 12, "customer": "Acme Industrial",
    },
    "SHP-1002": {
        "id": "SHP-1002", "origin": "Rotterdam", "destination": "Newark",
        "status": "at_port", "eta": "2026-04-28",
        "carrier": "MSC", "containers": 6, "customer": "Belden Foods",
    },
    "SHP-1003": {
        "id": "SHP-1003", "origin": "Busan", "destination": "Oakland",
        "status": "delayed", "eta": "2026-05-10",
        "carrier": "ONE", "containers": 18, "customer": "Acme Industrial",
    },
}
print(f"Seeded {len(SHIPMENT_DB)} shipments.")

## 6. Define the agent's tools

We define four OpenAI Agents SDK function tools:

| Tool | Read/Write | Purpose |
|---|---|---|
| `list_shipments` | Read | Enumerate all known shipments |
| `get_shipment_status` | Read | Look up the current state of one shipment |
| `update_shipment` | **Write** | Mutate a field on a shipment |
| `recall_operational_notes` | Read | Semantic search over Oracle-backed memory for prior context |

Each tool returns concise text for the model. Durable operational changes are also written back to Oracle AI Agent Memory.


In [ ]:
import json
from agents import function_tool


@function_tool
def list_shipments() -> str:
    """List every shipment currently tracked by the system."""
    summaries = [
        {"id": s["id"], "status": s["status"], "eta": s["eta"], "customer": s["customer"]}
        for s in SHIPMENT_DB.values()
    ]
    return json.dumps(summaries, indent=2)


@function_tool
def get_shipment_status(shipment_id: str) -> str:
    """Return the full record for one shipment by its ID, such as SHP-1001."""
    if shipment_id not in SHIPMENT_DB:
        return f"No shipment with id {shipment_id}."
    return json.dumps(SHIPMENT_DB[shipment_id], indent=2)


@function_tool
def update_shipment(shipment_id: str, field: str, value: str) -> str:
    """Update one field of one shipment and record the change as durable memory."""
    if shipment_id not in SHIPMENT_DB:
        return f"No shipment with id {shipment_id}."
    if field not in SHIPMENT_DB[shipment_id]:
        return f"Unknown field {field!r} on shipment {shipment_id}."
    before = SHIPMENT_DB[shipment_id][field]
    SHIPMENT_DB[shipment_id][field] = value
    memory_client.add_memory(
        f"Shipment {shipment_id} {field} changed from {before!r} to {value!r}.",
        user_id=OPERATOR_ID,
        agent_id=AGENT_ID,
        metadata={"shipment_id": shipment_id, "kind": "state_change"},
    )
    return f"Updated {shipment_id}.{field}: {before!r} -> {value!r}."


@function_tool
def recall_operational_notes(query: str, max_results: int = 5) -> str:
    """Search durable operational memory for prior notes relevant to the query."""
    results = memory_client.search(
        query,
        user_id=OPERATOR_ID,
        agent_id=AGENT_ID,
        max_results=max_results,
    )
    if not results:
        return "(no prior notes)"
    return "\n".join(f"- {r.content}  [distance={r.distance:.3f}]" for r in results)


print("Tools defined: list_shipments, get_shipment_status, update_shipment, recall_operational_notes")


## 7. Persist the agent session in Oracle AI Agent Memory


In [ ]:
import json
from typing import Optional
from agents.memory.session import SessionABC


class OracleAgentMemorySession(SessionABC):
    """Session backend that persists OpenAI Agents SDK items in Oracle AI Agent Memory."""

    def __init__(self, session_id: str, client: OracleAgentMemory, user_id: str, agent_id: str):
        self.session_id = session_id
        self._client = client
        self._store = client._store
        self._user_id = user_id
        self._agent_id = agent_id
        try:
            self._client.create_thread(thread_id=session_id, user_id=user_id, agent_id=agent_id)
        except ValueError:
            pass

    async def get_items(self, limit: Optional[int] = None) -> list:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=limit if limit else 10_000,
            metadata_filter={"session_item": True},
        )
        items = [json.loads(r.content) for r in records]
        return items[-limit:] if limit else items

    async def add_items(self, items: list) -> None:
        for item in items or []:
            self._client.add_memory(
                json.dumps(item),
                user_id=self._user_id,
                agent_id=self._agent_id,
                thread_id=self.session_id,
                metadata={"session_item": True},
            )

    async def pop_item(self) -> Optional[dict]:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=1,
            metadata_filter={"session_item": True},
        )
        if not records:
            return None
        last = records[-1]
        self._store.delete("memory", last.id)
        return json.loads(last.content)

    async def clear_session(self) -> None:
        records = self._store.list(
            "memory",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=10_000,
            metadata_filter={"session_item": True},
        )
        for record in records:
            self._store.delete("memory", record.id)


print("OracleAgentMemorySession implemented.")

SESSION_ID = "supply-session-001"
session = OracleAgentMemorySession(SESSION_ID, memory_client, OPERATOR_ID, AGENT_ID)
conv_thread = memory_client.get_thread(SESSION_ID)
print(f"Supply-chain session ready: {SESSION_ID}")


## 8. Configure the assistant with an OCI-backed model

The OpenAI Agents SDK expects a Chat Completions-shaped model client. The adapter below keeps that interface but sends every generation request to OCI Generative AI through the native OCI Python SDK.


In [ ]:
import asyncio
import json
import time
import uuid
from types import SimpleNamespace
from typing import Any

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from openai.types.chat import ChatCompletion, ChatCompletionChunk

set_tracing_disabled(True)


def _param_or_default(value: Any, default: Any) -> Any:
    if value is None:
        return default
    if value.__class__.__name__ in {"Omit", "NotGiven"}:
        return default
    return value


def _openai_tool_call_to_oci(models, tool_call: dict):
    function = tool_call.get("function", {}) if isinstance(tool_call, dict) else {}
    call = models.FunctionCall()
    if hasattr(models.FunctionCall, "TYPE_FUNCTION"):
        call.type = models.FunctionCall.TYPE_FUNCTION
    call.id = tool_call.get("id", "") if isinstance(tool_call, dict) else ""
    call.name = function.get("name", "")
    arguments = function.get("arguments", "{}")
    call.arguments = json.dumps(arguments) if isinstance(arguments, dict) else str(arguments or "{}")
    return call


def _openai_message_to_oci(models, message: dict):
    role = (message.get("role") or "user").lower()
    content_items = [_oci_text_content(models, message.get("content"))] if message.get("content") is not None else []

    if role == "system":
        oci_message = models.SystemMessage()
        oci_message.role = models.SystemMessage.ROLE_SYSTEM
    elif role == "assistant":
        oci_message = models.AssistantMessage()
        oci_message.role = models.AssistantMessage.ROLE_ASSISTANT
        if message.get("tool_calls"):
            oci_message.tool_calls = [
                _openai_tool_call_to_oci(models, tool_call)
                for tool_call in message.get("tool_calls", [])
            ]
    elif role == "tool":
        oci_message = models.ToolMessage()
        oci_message.role = models.ToolMessage.ROLE_TOOL
        oci_message.tool_call_id = message.get("tool_call_id", "")
    else:
        oci_message = models.UserMessage()
        oci_message.role = models.UserMessage.ROLE_USER

    oci_message.content = content_items
    return oci_message


def _openai_tool_to_oci(models, tool: dict):
    function = tool.get("function", {}) if isinstance(tool, dict) else {}
    definition = models.FunctionDefinition()
    if hasattr(models.FunctionDefinition, "TYPE_FUNCTION"):
        definition.type = models.FunctionDefinition.TYPE_FUNCTION
    definition.name = function.get("name", "")
    definition.description = function.get("description", "")
    definition.parameters = function.get("parameters", {})
    return definition


def _openai_tool_choice_to_oci(models, tool_choice: Any):
    if not tool_choice or tool_choice == "auto":
        choice = models.ToolChoiceAuto()
        if hasattr(models.ToolChoiceAuto, "TYPE_AUTO"):
            choice.type = models.ToolChoiceAuto.TYPE_AUTO
        return choice
    if tool_choice == "none":
        choice = models.ToolChoiceNone()
        if hasattr(models.ToolChoiceNone, "TYPE_NONE"):
            choice.type = models.ToolChoiceNone.TYPE_NONE
        return choice
    if tool_choice == "required":
        choice = models.ToolChoiceRequired()
        if hasattr(models.ToolChoiceRequired, "TYPE_REQUIRED"):
            choice.type = models.ToolChoiceRequired.TYPE_REQUIRED
        return choice
    if isinstance(tool_choice, dict):
        function = tool_choice.get("function", {})
        if function.get("name"):
            choice = models.ToolChoiceFunction()
            if hasattr(models.ToolChoiceFunction, "TYPE_FUNCTION"):
                choice.type = models.ToolChoiceFunction.TYPE_FUNCTION
            choice.name = function["name"]
            return choice
    return None


def _extract_oci_tool_calls(message: Any) -> list[dict]:
    converted = []
    for index, call in enumerate(getattr(message, "tool_calls", None) or []):
        arguments = getattr(call, "arguments", "{}") or "{}"
        if not isinstance(arguments, str):
            arguments = json.dumps(arguments)
        converted.append({
            "id": getattr(call, "id", "") or f"oci-tool-call-{index}",
            "type": "function",
            "function": {
                "name": getattr(call, "name", "") or "",
                "arguments": arguments,
            },
        })
    return converted


def _chat_completion_from_oci(model: str, message: Any) -> ChatCompletion:
    content = _extract_oci_text(message)
    tool_calls = _extract_oci_tool_calls(message)
    assistant_message = {
        "role": "assistant",
        "content": None if tool_calls else content,
    }
    if tool_calls:
        assistant_message["tool_calls"] = tool_calls

    return ChatCompletion(
        id=f"oci-{uuid.uuid4()}",
        object="chat.completion",
        created=int(time.time()),
        model=model,
        choices=[{
            "index": 0,
            "finish_reason": "tool_calls" if tool_calls else "stop",
            "message": assistant_message,
        }],
    )


async def _chat_completion_as_stream(completion: ChatCompletion):
    choice = completion.choices[0]
    message = choice.message
    chunk_id = f"{completion.id}-chunk"
    created = int(time.time())

    if message.tool_calls:
        delta_tool_calls = []
        for index, call in enumerate(message.tool_calls):
            delta_tool_calls.append({
                "index": index,
                "id": call.id,
                "type": "function",
                "function": {
                    "name": call.function.name,
                    "arguments": call.function.arguments,
                },
            })
        yield ChatCompletionChunk(
            id=chunk_id,
            object="chat.completion.chunk",
            created=created,
            model=completion.model,
            choices=[{
                "index": 0,
                "delta": {"role": "assistant", "tool_calls": delta_tool_calls},
                "finish_reason": None,
            }],
        )
        finish_reason = "tool_calls"
    else:
        yield ChatCompletionChunk(
            id=chunk_id,
            object="chat.completion.chunk",
            created=created,
            model=completion.model,
            choices=[{
                "index": 0,
                "delta": {"role": "assistant", "content": message.content or ""},
                "finish_reason": None,
            }],
        )
        finish_reason = "stop"

    yield ChatCompletionChunk(
        id=chunk_id,
        object="chat.completion.chunk",
        created=created,
        model=completion.model,
        choices=[{
            "index": 0,
            "delta": {},
            "finish_reason": finish_reason,
        }],
    )


class OciNativeChatCompletions:
    """Async Chat Completions-shaped adapter backed by OCI GenAI native Chat API."""

    def __init__(self, client, compartment_id: str, model_id: str, default_model: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id
        self.default_model = default_model

    async def create(
        self,
        *,
        model: str | None = None,
        messages: list[dict] | None = None,
        tools: list[dict] | None = None,
        tool_choice: Any = None,
        stream: bool = False,
        **kwargs,
    ):
        completion = await asyncio.to_thread(
            self._create_sync,
            model or self.default_model,
            messages or [],
            tools or [],
            tool_choice,
            kwargs,
        )
        if stream:
            return _chat_completion_as_stream(completion)
        return completion

    def _create_sync(self, model: str, messages: list[dict], tools: list[dict], tool_choice: Any, kwargs: dict):
        from oci.generative_ai_inference import models

        chat_request = models.GenericChatRequest()
        chat_request.api_format = models.BaseChatRequest.API_FORMAT_GENERIC
        chat_request.messages = [_openai_message_to_oci(models, message) for message in messages]
        chat_request.max_tokens = _param_or_default(
            kwargs.get("max_tokens"),
            _param_or_default(kwargs.get("max_completion_tokens"), 1200),
        )
        chat_request.temperature = _param_or_default(kwargs.get("temperature"), 0.2)
        chat_request.top_p = _param_or_default(kwargs.get("top_p"), 1.0)
        chat_request.top_k = _param_or_default(kwargs.get("top_k"), 0)
        chat_request.is_stream = False

        if tools:
            chat_request.tools = [_openai_tool_to_oci(models, tool) for tool in tools]
            native_tool_choice = _openai_tool_choice_to_oci(models, tool_choice)
            if native_tool_choice:
                chat_request.tool_choice = native_tool_choice

        serving_mode = models.OnDemandServingMode()
        serving_mode.model_id = self.model_id

        chat_detail = models.ChatDetails()
        chat_detail.serving_mode = serving_mode
        chat_detail.chat_request = chat_request
        chat_detail.compartment_id = self.compartment_id

        response = self.client.chat(chat_detail)
        data = response.data
        chat_response = getattr(data, "chat_response", data)
        choices = getattr(chat_response, "choices", None) or []
        message = getattr(choices[0], "message", None) if choices else None
        return _chat_completion_from_oci(model, message)


class OciNativeAsyncOpenAI:
    """Enough of AsyncOpenAI's shape for OpenAIChatCompletionsModel."""

    def __init__(self, native_client, endpoint: str, compartment_id: str, model_id: str, default_model: str):
        self.base_url = endpoint
        self.chat = SimpleNamespace(
            completions=OciNativeChatCompletions(
                native_client,
                compartment_id=compartment_id,
                model_id=model_id,
                default_model=default_model,
            )
        )

    def with_options(self, **_kwargs):
        return self


def create_agent_model():
    openai_shaped_client = OciNativeAsyncOpenAI(
        genai_client,
        endpoint=OCI_GENAI_ENDPOINT,
        compartment_id=OCI_GENAI_COMPARTMENT_OCID,
        model_id=OCI_GENAI_MODEL_OCID,
        default_model=LLM_MODEL,
    )
    return OpenAIChatCompletionsModel(model=LLM_MODEL, openai_client=openai_shaped_client)

from agents import Agent

SYSTEM_PROMPT = """You are a supply chain assistant for the West Coast logistics desk.

Operating rules:
1. When the operator asks about a shipment, use `get_shipment_status` or `list_shipments` first rather than speculating from context.
2. Before changing state, confirm the intended change in plain English, then call `update_shipment`.
3. When the operator mentions a customer, carrier, or prior event, call `recall_operational_notes` to check for relevant prior context.
4. Be concise. Surface concrete shipment IDs, fields, and values. Avoid restating what the tool returned; interpret it for the operator.
"""

supply_agent = Agent(
    name="SupplyChainAssistant",
    instructions=SYSTEM_PROMPT,
    tools=[list_shipments, get_shipment_status, update_shipment, recall_operational_notes],
    model=create_agent_model(),
)
print(f"Agent created: {supply_agent.name} using {LLM_PROVIDER}/{LLM_MODEL}")


## 9. Run a multi-turn conversation with `Runner`


In [ ]:
from agents import Runner
from oracleagentmemory.apis.thread import Message

operator_turns = [
    "What shipments are in flight right now?",
    "What's going on with SHP-1003? The customer just asked for an update.",
    "The carrier says SHP-1003 will now arrive on May 14 because of port congestion. Update the ETA.",
    "Remind me who the customer is on SHP-1003, and flag anything you know about them.",
]

for turn_idx, prompt in enumerate(operator_turns, 1):
    print(f"\n{'=' * 70}\nTurn {turn_idx} - OPERATOR: {prompt}\n{'=' * 70}")
    result = await Runner.run(supply_agent, prompt, session=session)
    answer = result.final_output or ""
    print(f"ASSISTANT: {answer}")
    conv_thread.add_messages([
        Message(role="user", content=prompt),
        Message(role="assistant", content=answer),
    ])


## 10. Inspect the memory state

Two things landed in Oracle during the conversation:

1. **Operational memories** written by `update_shipment` (state changes) and by the automatic extractor (from the conversation thread).
2. **Conversation messages** on the thread, which also feed the extractor to create further durable memories.

Let's look at both.

In [ ]:
# Durable memories relevant to shipment 1003
hits = memory_client.search(
    "SHP-1003 ETA change", user_id=OPERATOR_ID, agent_id=AGENT_ID, max_results=10,
)
print(f"Durable memories related to SHP-1003: {len(hits)}")
for r in hits:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

# Raw thread log
messages = conv_thread.get_messages()
print(f"\nConversation messages on thread: {len(messages)}")
for m in messages[:4]:
    print(f"  [{m.role}] {m.content[:100]}...")

# Confirm the live shipment record was mutated too
print(f"\nCurrent SHP-1003 ETA: {SHIPMENT_DB['SHP-1003']['eta']}")

## 11. Verify continuity — operator returns later

A later session should pick up the operational context without the operator re-explaining. We create a fresh session and ask a question that relies on memory from the earlier turn.


In [ ]:
followup = "Earlier we revised an ETA for a shipment headed to Oakland. What's the revised ETA and why did it change?"
fresh_session = OracleAgentMemorySession("supply-session-002", memory_client, OPERATOR_ID, AGENT_ID)

print(f"OPERATOR (new session): {followup}\n")
result = await Runner.run(supply_agent, followup, session=fresh_session)
print(f"ASSISTANT: {result.final_output}")


## 12. Cleanup

In [ ]:
# Uncomment to delete the conversation thread and its messages
# memory_client.delete_thread(conv_thread.thread_id)

connection.close()
print("Connection closed.")

## Key Takeaways

> **1. OCI can sit behind the agent interface.** The notebook keeps a modern tool-calling agent interface while generation and embeddings run through OCI Generative AI.

> **2. Tools should return concise text.** The model sees the tool return value on subsequent turns, so return structured but compact summaries instead of raw dumps.

> **3. Assistants need two memory tiers too.** Short-term context comes from the agent session. Durable operational context needs explicit writes through memory tools plus extraction from thread messages.

> **4. Every write that matters should hit memory, not just the live store.** The `update_shipment` tool mutates the live shipment record and writes a memory record describing the change.

> **5. Oracle AI Database unifies the substrate.** The live shipment store, durable memories, and conversation thread can all live in the same database.
